# Train Mamba on Z24 (10,000-point samples)
This notebook clones the project, downloads only `inputs.npy` and `labels.npy` from Hugging Face, then runs the medium temporal-holdout Mamba experiment: windows 0–2 train, window 3 validation, window 4 unused as a gap, and window 5 test. Every class and setup occurs in all splits; evaluation uses temporally separated windows from the same source recordings. Enable **GPU** and **Internet** in Kaggle before running.

In [ ]:
%cd /kaggle/working
!rm -rf time-series-project
!git clone https://github.com/TuanHungGiang/time-series-project.git
%cd time-series-project
!pip install -q huggingface_hub scikit-learn matplotlib

from huggingface_hub import snapshot_download

DATA_DIR = snapshot_download(
    repo_id="thanglexuan/Z24-dataset-processed",
    repo_type="dataset",
    local_dir="Z24-dataset-processed",
    allow_patterns=["inputs.npy", "labels.npy"],
)
print("Dataset saved at:", DATA_DIR)

In [ ]:
!python -u train_mamba.py --split-mode temporal_holdout --no-augment --epochs 100 --patience 20 --batch-size 8 --learning-rate 3e-4 --num-workers 4 --run-name mamba_temporal_holdout_noaug_seed42

In [ ]:
from pathlib import Path
from IPython.display import Image, display

run_dir = Path("results/mamba_temporal_holdout_noaug_seed42")
print((run_dir / "metrics.json").read_text())
for name in ["training_curves.png", "confusion_matrix.png", "accuracy_per_class.png",
             "accuracy_per_class_by_setup.png", "roc_curve.png",
             "tsne_epoch_001/tsne_2d.png", "tsne_epoch_final/tsne_2d.png"]:
    display(Image(filename=str(run_dir / name)))